# 02 · Turbulence Audit
Manual tau replication, chi2(N) calibration, decomposition, vol-standardization.
Use the parameter widgets below to change estimation settings without restarting.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from scipy import stats as sp_stats
from statsmodels.distributions.empirical_distribution import ECDF
from data.synthetic import generate_em_universe
from modules.turbulence import compute_turbulence_index, crisis_episodes
from core.covariance import SlowCovariance, VolStandardizer
print('Imports OK')


### Parameters (interactive)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_start  = widgets.DatePicker(description='Start date',
               value=__import__('datetime').date(2015, 1, 1))
w_end    = widgets.DatePicker(description='End date',
               value=__import__('datetime').date(2024, 12, 31))
w_window = widgets.IntSlider(min=63, max=504, step=21, value=252,
               description='Window', style={'description_width':'initial'})
w_slow_w = widgets.IntSlider(min=126, max=756, step=63, value=504,
               description='Slow window', style={'description_width':'initial'})
w_lam    = widgets.FloatSlider(min=0.90, max=0.99, step=0.01, value=0.94,
               description='EWMA lambda', style={'description_width':'initial'},
               readout_format='.2f')
w_vs     = widgets.Checkbox(value=True, description='Vol-standardize turbulence')

display(widgets.VBox([
    widgets.HBox([w_start, w_end]),
    w_window, w_slow_w, w_lam, w_vs,
]))
print('Adjust parameters above, then run the next cell.')


In [ ]:
# Read current widget values — run this after adjusting sliders
START   = str(w_start.value)
END     = str(w_end.value)
WINDOW  = w_window.value
SLOW_W  = w_slow_w.value
LAM     = w_lam.value
VOL_STD = w_vs.value

print(f'Parameters: start={START}  end={END}  window={WINDOW}')
print(f'            slow_w={SLOW_W}  lam={LAM:.2f}  vol_std={VOL_STD}')


### Data & Computation


In [ ]:
uni    = generate_em_universe(seed=42, start=START)
panel  = uni.panel
fx_ret = uni.fx
eq_ret = uni.equity

kw = dict(window=WINDOW, min_periods=60, vol_standardize=VOL_STD, slow_window=SLOW_W)
turb_vs   = compute_turbulence_index(panel,  **kw)
turb_raw  = compute_turbulence_index(panel,  **{**kw, 'vol_standardize': False})
turb_fx   = compute_turbulence_index(fx_ret, **kw)
turb_eq   = compute_turbulence_index(eq_ret, **kw)

print(f'Panel shape : {panel.shape}')
print(f'Regime (vol-std): {turb_vs.current_regime()}  score: {turb_vs.current_score():.2f}')


## 2 · Manual tau Replication
tau_t = (r_t - mu)' Sigma^{-1} (r_t - mu),  Sigma = SLOW-track covariance.


In [ ]:
TARGET = '2020-03-20'
loc = panel.index.get_indexer([TARGET], method='nearest')[0]
audit_idx = panel.index[loc]
print(f'Audit date: {audit_idx.date()}')

window_data = panel.iloc[max(0, loc-SLOW_W):loc].dropna(how='any')
cov_result  = SlowCovariance().fit(window_data, window=SLOW_W)
mu          = cov_result.mu
Sigma_inv   = cov_result.sigma_inv

vol_std_panel = VolStandardizer().fit_transform(panel).fillna(0.0)
r_t = vol_std_panel.loc[audit_idx].values if VOL_STD else panel.loc[audit_idx].values

tau_manual = float((r_t - mu) @ Sigma_inv @ (r_t - mu))
tau_module = float(turb_vs.turbulence.loc[audit_idx])
replication_df = pd.DataFrame([{'Date': str(audit_idx.date()),
    'tau_manual': round(tau_manual,4), 'tau_module': round(tau_module,4),
    'abs_diff': round(abs(tau_manual - tau_module),6)}])
print(replication_df.to_string(index=False))
print(f'Regime: {turb_vs.regime.loc[audit_idx]}')


## 3 · chi2(N) Calibration Plot


In [ ]:
N = panel.shape[1]
CALM_MASK = ((panel.index >= '2015-01-01') & (panel.index < '2018-06-01')) | \
            ((panel.index >= '2021-01-01') & (panel.index < '2022-01-01'))
tau_calm = turb_vs.turbulence.loc[CALM_MASK].dropna()

x = np.linspace(0, float(tau_calm.quantile(0.999)), 300)
chi2_cdf = sp_stats.chi2.cdf(x, df=N)
emp_cdf  = ECDF(tau_calm.values)(x)
calibration_df = pd.DataFrame({'x': x.round(4), 'empirical_cdf': emp_cdf.round(4),
                                'chi2_cdf': chi2_cdf.round(4)})

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, chi2_cdf, 'r--', lw=1.5, label=f'Theoretical chi2({N})')
ax.plot(x, emp_cdf,  'b-',  lw=1.5, label='Empirical (calm)')
ax.axvline(sp_stats.chi2.ppf(0.95, N), color='orange', ls=':', alpha=0.7, label='chi2(0.95)')
ax.axvline(sp_stats.chi2.ppf(0.99, N), color='red',    ls=':', alpha=0.7, label='chi2(0.99)')
ax.set_xlabel('tau'); ax.set_ylabel('CDF')
ax.set_title(f'Empirical vs chi2({N}) CDF — calm-period tau')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()
print(f'Calm tau mean={tau_calm.mean():.2f}  chi2({N}) mean={N}')


## 4 · Turbulence Time Series with Regime Shading


In [ ]:
REGIME_HEX = {'Calm':'#2ecc71','Elevated':'#f39c12','Turbulent':'#e67e22','Crisis':'#e74c3c'}
fig, ax = plt.subplots(figsize=(14, 4))
ts  = turb_vs.turbulence.dropna()
reg = turb_vs.regime.reindex(ts.index)
for label, col in REGIME_HEX.items():
    mask = reg == label
    if not mask.any(): continue
    in_block, x0 = False, None
    for dt, val in mask.items():
        if val and not in_block:  x0, in_block = dt, True
        elif not val and in_block: ax.axvspan(x0, dt, alpha=0.25, color=col, lw=0); in_block = False
    if in_block: ax.axvspan(x0, mask.index[-1], alpha=0.25, color=col, lw=0)
ax.plot(ts.index, ts.values, color='#00d4aa', lw=0.9)
for thr, label in [('elevated','Elevated'),('turbulent','Turbulent'),('crisis','Crisis')]:
    ax.axhline(turb_vs.thresholds[thr], color=REGIME_HEX[label], ls='--', lw=0.8, alpha=0.7)
patches = [mpatches.Patch(color=REGIME_HEX[l], alpha=0.5, label=l) for l in REGIME_HEX]
ax.legend(handles=patches, loc='upper left', fontsize=8)
ax.set_ylabel('tau (vol-std)'); ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title(f'Turbulence Index (window={WINDOW}, slow_w={SLOW_W})')
plt.tight_layout(); plt.show()


## 5 · Magnitude vs Correlation Decomposition


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
mag  = turb_vs.magnitude_component.reindex(ts.index)
corr = turb_vs.correlation_component.reindex(ts.index)
axes[0].fill_between(ts.index, mag,  alpha=0.7, color='#3498db', label='Magnitude')
axes[0].fill_between(ts.index, corr, alpha=0.7, color='#9b59b6', label='Correlation')
axes[0].set_ylabel('Component tau'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.15)
axes[0].set_title('Turbulence Decomposition')
total    = (mag + corr).clip(lower=1e-8)
frac_cor = corr.clip(lower=0) / total
axes[1].plot(frac_cor.index, frac_cor.rolling(21).mean(), color='#9b59b6', lw=1.1)
axes[1].axhline(0.5, color='white', ls='--', lw=0.8, alpha=0.4)
axes[1].set_ylabel('Fraction from correlation (21d MA)'); axes[1].grid(alpha=0.15)
axes[1].set_ylim(0,1)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()


## 6 · Crisis Episode Table


In [ ]:
crisis_df   = crisis_episodes(turb_vs, regime='Crisis',   min_duration_days=3)
turbulent_df = crisis_episodes(turb_vs, regime='Turbulent', min_duration_days=5)
for label, eps in [('Crisis', crisis_df), ('Turbulent', turbulent_df)]:
    print(f'{label} episodes: {len(eps)}')
    if not eps.empty:
        eps = eps.copy()
        eps['start'] = pd.to_datetime(eps['start']).dt.strftime('%Y-%m-%d')
        eps['end']   = pd.to_datetime(eps['end']).dt.strftime('%Y-%m-%d')
        eps.columns  = ['Start','End','Duration (d)','Peak tau','Mean tau']
        print(eps.round(2).to_string(index=False))
    print()


## 7 · Vol-Standardization Effect


In [ ]:
loc_spike  = panel.index.get_indexer(['2020-03-20'], method='nearest')[0]
spike_date = panel.index[loc_spike]
panel_spiked = panel.copy(); panel_spiked.loc[spike_date] *= 5.0
t_vs  = compute_turbulence_index(panel_spiked, **kw)
t_raw = compute_turbulence_index(panel_spiked, **{**kw, 'vol_standardize': False})
print(f'Spike date : {spike_date.date()}')
print(f'  vol-std  tau={t_vs.turbulence.loc[spike_date]:.1f}  regime={t_vs.regime.loc[spike_date]}')
print(f'  raw      tau={t_raw.turbulence.loc[spike_date]:.1f}  regime={t_raw.regime.loc[spike_date]}')
if t_vs.regime.loc[spike_date] != 'Crisis':
    print('OK: vol-standardization suppressed the pure vol spike')
else:
    print('NOTE: spike still Crisis — EWMA adaptation may need more time')


## 8 · Export to Excel


In [ ]:
from datetime import date
os.makedirs('../exports', exist_ok=True)
filename = f'../exports/02_turbulence_audit_{date.today()}.xlsx'
with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    turb_vs.to_frame().to_excel(writer, sheet_name='Turbulence Panel')
    turb_fx.to_frame().to_excel(writer, sheet_name='Turbulence FX')
    turb_eq.to_frame().to_excel(writer, sheet_name='Turbulence Equity')
    if not crisis_df.empty: crisis_df.to_excel(writer, sheet_name='Crisis Episodes', index=False)
    replication_df.to_excel(writer, sheet_name='Manual Replication', index=False)
    calibration_df.to_excel(writer, sheet_name='Chi2 Calibration', index=False)
print(f'Exported: {filename}')
